# Experiment 2: NMF Topic Modeling

This notebook runs a NMF pipeline using preprocessed input from the data preprocessing stage.

## Hypothesis:  
Similar to the LDA model, the NMF model itself has certain hypothesis:  
H1 - In NMF, the TF-IDF matrix is ​​used for calculation. We assume that the words most meaningful for distinguishing documents determine the semantics.  
H2 - NMF is essentially a geometric decomposition and approximation, and the resulting weights are not strictly equal to the probabilities of topics. This assumption can be seen as an attempt to enhance business understanding and LDA comparability.

## Pipeline: 

**load data - TfidfVectorizer - NMF - topic export**

In [1]:
import importlib
from pathlib import Path
import sys
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

PROJECT_ROOT = Path.cwd().parents[2]
axis2_dir = PROJECT_ROOT / 'notebooks' / '03_Topic_and_Insights'
sys.path.insert(0, str(axis2_dir))
from Data_preprocessing import Parameters_Path as config

In [2]:
importlib.reload(config)

INPUT_PATH  = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'Experiment_2_NMF'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NMF_TOPICS_PATH = RESULTS_DIR / 'Experiment_2_nmf_topics.json'
DOC_TOPIC_PATH  = RESULTS_DIR / 'Experiment_2_nmf_doc_topics.csv'

tfidf_cfg = config.PARAMETERS['tfidf_vectorizer']
nmf_cfg   = config.PARAMETERS['nmf']
N_COMPONENTS = int(nmf_cfg['n_components'])
N_TOP_WORDS  = int(nmf_cfg['n_top_words'])

In [3]:
df = pd.read_csv(INPUT_PATH)
texts = df['text_cleaned_axis1'].astype(str).str.strip()
texts = texts[texts.ne('')]

In [4]:
vectorizer = TfidfVectorizer(
    max_df=tfidf_cfg['max_df'],
    min_df=tfidf_cfg['min_df'],
    max_features=tfidf_cfg['max_features'],
    stop_words=tfidf_cfg['stop_words']
)
X_tfidf = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

nmf = NMF(
    n_components=N_COMPONENTS,
    random_state=nmf_cfg['random_state'],
    max_iter=nmf_cfg['max_iter']
)
doc_topic = nmf.fit_transform(X_tfidf)

print('TfidfVectorizer config:', tfidf_cfg)
print('NMF config:', nmf_cfg)

TfidfVectorizer config: {'max_df': 0.8, 'min_df': 2, 'max_features': 1200, 'stop_words': 'english'}
NMF config: {'n_components': 4, 'n_top_words': 8, 'random_state': 42, 'max_iter': 500}


In [5]:
def extract_topics(model, vocab, n_top_words=8):
    topics = {}
    for i, topic in enumerate(model.components_, start=1):
        idx = topic.argsort()[-n_top_words:][::-1]
        topics[f'topic_{i}'] = {
            'keywords': [vocab[j] for j in idx],
            'weights':  topic[idx].tolist()
        }
    return topics

nmf_topics = extract_topics(nmf, feature_names, N_TOP_WORDS)

with open(NMF_TOPICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(nmf_topics, f, indent=2, ensure_ascii=False)

# row-normalise so max_topic_prob is on [0,1], comparable to LDA
doc_topic_norm = doc_topic / (doc_topic.sum(axis=1, keepdims=True))

topic_cols = [f'nmf_topic_{i}' for i in range(1, N_COMPONENTS + 1)]
doc_topic_df  = pd.DataFrame(doc_topic_norm, columns=topic_cols)
doc_topic_df['nmf_dominant_topic'] = doc_topic_df[topic_cols].values.argmax(axis=1) + 1
doc_topic_df['max_topic_prob'] = doc_topic_df[topic_cols].values.max(axis=1)
doc_topic_df.to_csv(DOC_TOPIC_PATH, index=False, encoding='utf-8')

In [6]:
print('NMF topics preview:')
for topic_name, content in nmf_topics.items():
    print(f"{topic_name}: {', '.join(content['keywords'])}")

NMF topics preview:
topic_1: software, data, inquiry, refund, started, device, cancellation, application
topic_2: account, access, reset, password, login, regain, credentials, invalid
topic_3: screen, message, mean, popping, peculiar, says, error, refund
topic_4: action, perform, desired, guide, option, unable, bug, fixes
